In [1]:
#cell 1
# Check GPU.
!nvidia-smi

Tue Jun 16 07:26:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# cell 2
# Install dependencies for A100 80GB.
# This cell replaces the old Blackwell/CUDA 13.0/cu130 installation cell.

!pip -q install -U uv

# Basic dependencies.
!uv pip install --system -U openai tqdm requests psutil pandas

# Install a recent vLLM build for Qwen3.5.
# For A100, do not force cu130/Blackwell. Let uv select the correct CUDA/PyTorch backend.
!uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

# Version check.
import sys
import torch
import vllm

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("GPU memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)

Using Python 3.12.13 environment at: /usr
Resolved 24 packages in 120ms
Prepared 1 package in 0.39ms
Uninstalled 1 package in 15ms
Installed 1 package in 18ms
 - numpy==2.3.5
 + numpy==2.4.6
Using Python 3.12.13 environment at: /usr
Resolved 189 packages in 13.41s
Prepared 1 package in 0.44ms
Uninstalled 1 package in 14ms
Installed 1 package in 17ms
 - numpy==2.4.6
 + numpy==2.3.5
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.11.0+cu130
Torch CUDA: 13.0
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU memory GB: 79.25
vLLM: 0.23.1rc1.dev49+ga7fdfeef7


In [3]:
# cell 3
# Mount Google Drive and prepare BM25 input/output paths.
from google.colab import drive
from pathlib import Path
import shutil
import json
import os

drive.mount("/content/drive")

# Main BM25 folder that contains all answer files.
GDRIVE_INPUT_DIR = Path("/content/drive/MyDrive/final_project/BM25/answer")

# Six BM25 answer files to evaluate.
# The file naming is not fully uniform:
# - gemma4 and gpt_oss_120b: dataset_bm25_llm_answers.json
# - qwen3.5: dataset_qwen3.5_bm25_answers.json
INPUT_FILES = [
    "hotpotqa_bm25_gemma4_answers.json",
    "hotpotqa_bm25_gpt_oss_120b_answers.json",
    "hotpotqa_qwen3.5_bm25_answers.json",

    "2wikimultihopqa_bm25_gemma4_answers.json",
    "2wikimultihopqa_bm25_gpt_oss_120b_answers.json",
    "2wikimultihopqa_qwen3.5_bm25_answers.json",
]

# Separate dataset groups.
# These will be used later in two separate run/save cells.
HOTPOTQA_INPUT_FILES = [
    file_name for file_name in INPUT_FILES
    if file_name.startswith("hotpotqa_")
]

TWIKIMULTIHOPQA_INPUT_FILES = [
    file_name for file_name in INPUT_FILES
    if file_name.startswith("2wikimultihopqa_")
]

# Local working directory.
LOCAL_WORK_DIR = Path("/content/bm25_accuracy_llm_judge")
LOCAL_INPUT_DIR = LOCAL_WORK_DIR / "inputs"
LOCAL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output directory for judged results and accuracy summaries.
# Results will be saved inside the BM25 answer folder.
GDRIVE_OUTPUT_DIR = GDRIVE_INPUT_DIR / "bm25" / "llm_judge_qwen35_27b_accuracy_a100"
GDRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def parse_input_file_name(file_name: str) -> dict:
    # Parse dataset, retrieval method, and LLM name from a BM25 answer file name.
    stem = Path(file_name).stem

    if stem.endswith("_answers"):
        base = stem[:-len("_answers")]
    else:
        base = stem

    dataset_name, rest = base.split("_", 1)
    parts = rest.split("_")

    if "bm25" not in parts:
        raise ValueError(f"Could not find bm25 marker in file name: {file_name}")

    retrieval_name = "bm25"

    # Remove only the bm25 marker and keep the remaining parts as the LLM name.
    # Examples:
    # bm25_gemma4 -> gemma4
    # bm25_gpt_oss_120b -> gpt_oss_120b
    # qwen3.5_bm25 -> qwen3.5
    llm_name = "_".join([part for part in parts if part != "bm25"])

    return {
        "dataset": dataset_name,
        "retrieval": retrieval_name,
        "llm": llm_name,
    }

FILE_METADATA = {
    file_name: parse_input_file_name(file_name)
    for file_name in INPUT_FILES
}

# Copy files to local disk.
for file_name in INPUT_FILES:
    src = GDRIVE_INPUT_DIR / file_name
    dst = LOCAL_INPUT_DIR / file_name

    assert src.exists(), f"Input file not found: {src}"

    shutil.copy2(src, dst)
    print("Copied:", src, "->", dst)
    print("Metadata:", FILE_METADATA[file_name])

print("HotpotQA files:", HOTPOTQA_INPUT_FILES)
print("2WikiMultiHopQA files:", TWIKIMULTIHOPQA_INPUT_FILES)
print("Local input dir:", LOCAL_INPUT_DIR)
print("Output dir:", GDRIVE_OUTPUT_DIR)

Mounted at /content/drive
Copied: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_bm25_gemma4_answers.json -> /content/bm25_accuracy_llm_judge/inputs/hotpotqa_bm25_gemma4_answers.json
Metadata: {'dataset': 'hotpotqa', 'retrieval': 'bm25', 'llm': 'gemma4'}
Copied: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_bm25_gpt_oss_120b_answers.json -> /content/bm25_accuracy_llm_judge/inputs/hotpotqa_bm25_gpt_oss_120b_answers.json
Metadata: {'dataset': 'hotpotqa', 'retrieval': 'bm25', 'llm': 'gpt_oss_120b'}
Copied: /content/drive/MyDrive/final_project/BM25/answer/hotpotqa_qwen3.5_bm25_answers.json -> /content/bm25_accuracy_llm_judge/inputs/hotpotqa_qwen3.5_bm25_answers.json
Metadata: {'dataset': 'hotpotqa', 'retrieval': 'bm25', 'llm': 'qwen3.5'}
Copied: /content/drive/MyDrive/final_project/BM25/answer/2wikimultihopqa_bm25_gemma4_answers.json -> /content/bm25_accuracy_llm_judge/inputs/2wikimultihopqa_bm25_gemma4_answers.json
Metadata: {'dataset': '2wikimultihopqa', 'retrieval

In [4]:
#cell 4
# Load input JSON files.
from collections import Counter

def load_json_list(path: Path):
    # Load a JSON list.
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    assert isinstance(data, list), f"Expected a list in {path}"
    return data

all_input_data = {}

for file_name in INPUT_FILES:
    path = LOCAL_INPUT_DIR / file_name
    data = load_json_list(path)
    all_input_data[file_name] = data

    print("\nFile:", file_name)
    print("Dataset:", FILE_METADATA[file_name]["dataset"])
    print("LLM:", FILE_METADATA[file_name]["llm"])
    print("Rows:", len(data))
    print("Types:", Counter(x.get("type") for x in data))

    if len(data) > 0:
        print("Keys:", sorted(data[0].keys()))


File: hotpotqa_bm25_gemma4_answers.json
Dataset: hotpotqa
LLM: gemma4
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['gt', 'question', 'response', 'type']

File: hotpotqa_bm25_gpt_oss_120b_answers.json
Dataset: hotpotqa
LLM: gpt_oss_120b
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['gt', 'question', 'response', 'type']

File: hotpotqa_qwen3.5_bm25_answers.json
Dataset: hotpotqa
LLM: qwen3.5
Rows: 1000
Types: Counter({'bridge': 700, 'comparison': 300})
Keys: ['gt', 'question', 'response', 'type']

File: 2wikimultihopqa_bm25_gemma4_answers.json
Dataset: 2wikimultihopqa
LLM: gemma4
Rows: 1000
Types: Counter({'bridge_comparison': 250, 'compositional': 250, 'comparison': 250, 'inference': 250})
Keys: ['gt', 'question', 'response', 'type']

File: 2wikimultihopqa_bm25_gpt_oss_120b_answers.json
Dataset: 2wikimultihopqa
LLM: gpt_oss_120b
Rows: 1000
Types: Counter({'bridge_comparison': 250, 'compositional': 250, 'comparison': 250, 'inference': 250})

In [5]:
# cell 5
# Start vLLM server for Qwen3.5-27B on A100 80GB.
import subprocess
import time
import requests
import shlex
import psutil
from pathlib import Path
import os

MODEL_NAME = "Qwen/Qwen3.5-27B"
PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# Conservative context length for a single A100 80GB.
# The judging prompt is short, so 8192 is usually enough and leaves more room for KV cache.
MAX_MODEL_LEN = 8192

# A100 80GB memory setting.
# If startup still has OOM, reduce this to 0.86.
GPU_MEMORY_UTILIZATION = 0.90

# Conservative throughput settings for single A100 80GB.
# If stable and you want more speed, try MAX_NUM_SEQS=8 later.
MAX_NUM_SEQS = 4
MAX_NUM_BATCHED_TOKENS = 8192

SERVER_LOG_PATH = Path("/content/vllm_server.log")
SERVER_PID_PATH = Path("/content/vllm_server.pid")

def kill_process_tree(pid):
    # Kill a process and its children.
    try:
        parent = psutil.Process(int(pid))

        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass

        parent.kill()
        parent.wait(timeout=10)
        print("Killed old process tree:", pid)

    except Exception:
        pass

# Stop old PID.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(old_pid)

# Kill leftover vLLM servers.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

time.sleep(3)

cmd = [
    "vllm", "serve", MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode frees memory because Qwen3.5 is a multimodal model.
    "--language-model-only",

    # Qwen3/Qwen3.5 reasoning parser.
    "--reasoning-parser", "qwen3",

    # Disable thinking mode by default for judge-style JSON outputs.
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    # Throughput settings.
    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    # Shared prompt prefix cache.
    "--enable-prefix-caching",

    # Use vLLM generation config.
    "--generation-config", "vllm",

    # A100 supports bfloat16.
    "--dtype", "bfloat16",

    # Safe for custom model code.
    "--trust-remote-code",
]

# Optional OOM fallback:
# If CUDA graph memory causes startup OOM, uncomment the next line.
# cmd.append("--enforce-eager")

server_env = os.environ.copy()

# Avoid FlashInfer sampler issues in some Colab/vLLM environments.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

# Do NOT set Blackwell-only CUDA 13.0 variables on A100.
server_env.pop("VLLM_MAIN_CUDA_VERSION", None)
server_env.pop("TORCH_CUDA_ARCH_LIST", None)

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 8192 --gpu-memory-utilization 0.9 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 4 --max-num-batched-tokens 8192 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=None
TORCH_CUDA_ARCH_LIST=None

Started vLLM server.
PID: 5819
Log: /content/vllm_server.log


In [6]:
#cell 6
# Wait for vLLM server.
import time
import requests
from pathlib import Path

def tail_log(path, n=80):
    # Return recent log lines.
    path = Path(path)

    if not path.exists():
        return ""

    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    return_code = proc.poll()

    if return_code is not None:
        print(f"vLLM process exited. Return code: {return_code}")
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError("vLLM server crashed or exited during startup.")

    try:
        health = requests.get(f"http://localhost:{PORT}/health", timeout=5)

        if health.status_code == 200:
            models = requests.get(f"{BASE_URL}/models", timeout=10)

            if models.status_code == 200:
                ready = True
                model_info = models.json()["data"][0]

                print("vLLM server is ready.")
                print("Model:", model_info["id"])
                print("Max model len:", model_info.get("max_model_len"))
                break

    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")

        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)

        print("-" * 80)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

Waiting... 0s
--------------------------------------------------------------------------------
Waiting... 60s
(APIServer pid=5819) INFO 06-16 07:27:45 [api_utils.py:273] non-default args: {'model_tag': 'Qwen/Qwen3.5-27B', 'default_chat_template_kwargs': {'enable_thinking': False}, 'host': '0.0.0.0', 'model': 'Qwen/Qwen3.5-27B', 'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 8192, 'generation_config': 'vllm', 'reasoning_parser': 'qwen3', 'gpu_memory_utilization': 0.9, 'enable_prefix_caching': True, 'language_model_only': True, 'max_num_batched_tokens': 8192, 'max_num_seqs': 4}
(APIServer pid=5819) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(APIServer pid=5819) INFO 06-16 07:28:01 [model.py:598] Resolved architecture: Qwen3_5ForConditionalGeneration
(APIServer pid=5819) INFO 06-16 07:28:01 [model.py:1723] Using max model len 8192
(APIServer pid=5819) INFO 06-16 07:28:01 [sched

In [7]:
#cell 7
# Define the judge prompt and JSON schema.
JUDGE_SYSTEM_PROMPT = """
You are a strict but fair answer judge.

You will receive:
- a question
- the ground-truth answer
- a candidate LLM response

Your job is to judge whether the candidate response correctly answers the question.
Use the ground-truth answer as the authoritative answer.
Do not solve the question yourself.
Do not add background knowledge.
Do not write analysis outside JSON.

Return only this JSON object, with reason first and score second:
{
  "reason": "short reason",
  "score": 0 or 1
}

Rules:
- Score 1 if the candidate gives the correct answer to the question.
- Score 1 if the candidate is semantically the same as the ground truth, even with different wording.
- Score 1 if the candidate gives the main requested answer clearly and any missing part is only a harmless clarifying suffix.
- Score 1 if the candidate includes extra information that does not contradict the correct answer.
- Score 0 if the candidate is wrong.
- Score 0 if the candidate is incomplete for what the question asks.
- Score 0 if the candidate is vague, too broad, or too narrow.
- Score 0 if the candidate says the answer is unknown, unavailable, cannot be determined, or similar.
- Score 0 if the candidate contains a contradiction or an incorrect final answer.
- Use the question to decide what details are required.
- Do not require exact string matching.
- When unsure, score 0.

Keep the reason short: maximum 25 words.
Do not use double quotation marks inside the reason string.
""".strip()

JUDGE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "reason": {
            "type": "string"
        },
        "score": {
            "type": "integer",
            "enum": [0, 1]
        },
    },
    "required": ["reason", "score"],
    "additionalProperties": False,
}

def build_judge_user_prompt(question: str, gt: str, response: str) -> str:
    # Build one judging prompt.
    return f"""
Question:
{question}

Ground-truth answer:
{gt}

Candidate LLM response:
{response}

Judge the candidate response.
Return only JSON with reason first and score second.
""".strip()

In [8]:
#cell 8
# Create OpenAI-compatible client and JSON parser.
from openai import OpenAI
from typing import Any, Dict
import json
import time
import re

client = OpenAI(
    base_url=BASE_URL,
    api_key="EMPTY",
)

def coerce_malformed_judge_json(text: str) -> Dict[str, Any] | None:
    # Recover judge outputs that look like JSON but contain unescaped quotes inside reason.
    text = (text or "").strip()

    score_match = re.search(r'"score"\s*:\s*"?([01])"?', text)

    if score_match is None:
        score_match = re.search(r"(?im)^\s*score\s*[:=]\s*([01])\s*$", text)

    if score_match is None:
        return None

    score = int(score_match.group(1))

    reason = None
    reason_marker = re.search(r'"reason"\s*:\s*"', text, flags=re.S)

    if reason_marker is not None:
        start = reason_marker.end()
        after = text[start:]

        # Prefer the quote immediately before the next score field.
        sep = re.search(r'"\s*,\s*"score"\s*:', after, flags=re.S)

        if sep is not None:
            reason = after[:sep.start()]
        else:
            # If reason is the final field, use the last quote before the closing brace.
            end = re.search(r'"\s*}\s*$', after, flags=re.S)
            reason = after[:end.start()] if end is not None else after

    if reason is None:
        reason = "Recovered malformed judge JSON."

    reason = reason.replace('\\"', '"')
    reason = reason.replace("\\n", " ")
    reason = reason.replace('"', "'")
    reason = " ".join(reason.split()).strip()

    if not reason:
        reason = "Recovered malformed judge JSON."

    return {
        "reason": reason,
        "score": score,
    }

def extract_first_json_object(text: str) -> Dict[str, Any]:
    # Extract the first JSON object.
    text = (text or "").strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    # Robust fallback for malformed JSON with unescaped quotes in reason.
    recovered = coerce_malformed_judge_json(text)
    if recovered is not None:
        return recovered

    start = text.find("{")

    if start == -1:
        raise ValueError(f"No JSON object found: {text[:300]}")

    depth = 0
    in_str = False
    escape = False

    for i in range(start, len(text)):
        ch = text[i]

        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
        else:
            if ch == '"':
                in_str = True
            elif ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1

                if depth == 0:
                    candidate = text[start:i + 1]

                    try:
                        return json.loads(candidate)
                    except Exception:
                        recovered = coerce_malformed_judge_json(candidate)

                        if recovered is not None:
                            return recovered

                        raise

    recovered = coerce_malformed_judge_json(text)

    if recovered is not None:
        return recovered

    raise ValueError(f"Incomplete JSON object: {text[:300]}")

def normalize_judge_output(obj: Dict[str, Any]) -> Dict[str, Any]:
    # Normalize judge output.
    score = obj.get("score")

    if isinstance(score, str):
        score = score.strip()
        if score in {"0", "1"}:
            score = int(score)

    if score not in {0, 1}:
        raise ValueError(f"Invalid score: {obj}")

    reason = str(obj.get("reason", "")).strip()

    if not reason:
        reason = "No reason provided."

    # Keep reason compact and avoid unsafe inner double quotes.
    reason = reason.replace('"', "'")
    reason = " ".join(reason.split())

    return {
        "reason": reason,
        "score": int(score),
    }

In [9]:
#cell 9
# Define one judge call with guided JSON and retries.
JUDGE_TEMPERATURE = 0.0
JUDGE_MAX_TOKENS = 512
MAX_RETRIES = 5

def judge_one_record(record: Dict[str, Any]) -> Dict[str, Any]:
    # Judge one row.
    question = str(record.get("question", ""))
    gt = str(record.get("gt", ""))
    response = str(record.get("response", ""))

    messages = [
        {
            "role": "system",
            "content": JUDGE_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_judge_user_prompt(question, gt, response),
        },
    ]

    last_error = None
    started = time.perf_counter()

    for attempt in range(MAX_RETRIES):
        try:
            # First choice: vLLM guided JSON.
            try:
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    temperature=JUDGE_TEMPERATURE,
                    max_tokens=JUDGE_MAX_TOKENS,
                    extra_body={
                        "guided_json": JUDGE_JSON_SCHEMA,
                    },
                )

            except Exception:
                # Fallback: OpenAI-style JSON mode.
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=messages,
                    temperature=JUDGE_TEMPERATURE,
                    max_tokens=JUDGE_MAX_TOKENS,
                    response_format={"type": "json_object"},
                )

            raw_text = completion.choices[0].message.content or ""
            parsed = normalize_judge_output(extract_first_json_object(raw_text))

            latency = time.perf_counter() - started

            return {
                "judge_reason": parsed["reason"],
                "judge_score": parsed["score"],
                "judge_raw": raw_text,
                "judge_error": None,
                "judge_latency_sec": latency,
            }

        except Exception as e:
            last_error = repr(e)
            time.sleep(2.0 * (attempt + 1))

    latency = time.perf_counter() - started

    return {
        "judge_reason": None,
        "judge_score": None,
        "judge_raw": None,
        "judge_error": last_error,
        "judge_latency_sec": latency,
    }

In [10]:
#cell 10
# Judge one file with resume support and automatic rejudging of failed rows.
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from pathlib import Path
from typing import Dict, Any
import json

REQUEST_CONCURRENCY = 16
FILE_MAX_ROUNDS = 5

# Keep the notebook moving so the repair cell can fix any remaining malformed outputs.
RAISE_ON_FAILED_JUDGEMENTS = False

def make_result_key(record: Dict[str, Any], row_id: int) -> str:
    # Make a stable key.
    if record.get("source_index") is not None:
        return str(record.get("source_index"))

    return str(row_id)

def is_valid_judgement(obj: Dict[str, Any] | None) -> bool:
    # Check if a saved judgement is usable.
    if obj is None:
        return False

    if obj.get("judge_error") is not None:
        return False

    if obj.get("judge_score") not in {0, 1}:
        return False

    return True

def load_existing_jsonl(path: Path) -> Dict[str, Dict[str, Any]]:
    # Load latest result for each key.
    existing = {}

    if not path.exists():
        return existing

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except Exception:
                continue

            key = str(obj.get("_judge_key"))
            existing[key] = obj

    return existing

def append_jsonl(path: Path, obj: Dict[str, Any]):
    # Append one JSONL row.
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

def judge_file(input_file_name: str) -> Path:
    # Judge all rows in one file.
    data = all_input_data[input_file_name]
    metadata = FILE_METADATA[input_file_name]

    stem = Path(input_file_name).stem
    out_jsonl = GDRIVE_OUTPUT_DIR / f"{stem}__judged.jsonl"
    out_json = GDRIVE_OUTPUT_DIR / f"{stem}__judged.json"

    for round_id in range(1, FILE_MAX_ROUNDS + 1):
        existing = load_existing_jsonl(out_jsonl)

        todo = []

        for row_id, record in enumerate(data):
            key = make_result_key(record, row_id)
            current = existing.get(key)

            # Rejudge missing or failed rows.
            if not is_valid_judgement(current):
                todo.append((row_id, key, record))

        print("\nFile:", input_file_name)
        print("Dataset:", metadata["dataset"])
        print("LLM:", metadata["llm"])
        print("Round:", round_id)
        print("Rows:", len(data))
        print("Valid already judged:", len(data) - len(todo))
        print("Remaining or failed:", len(todo))

        if not todo:
            break

        with ThreadPoolExecutor(max_workers=REQUEST_CONCURRENCY) as executor:
            futures = {
                executor.submit(judge_one_record, record): (row_id, key, record)
                for row_id, key, record in todo
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"Judging {input_file_name} | round {round_id}",
            ):
                row_id, key, record = futures[future]
                judge_result = future.result()

                out_obj = {
                    "_judge_key": key,
                    "_row_id": row_id,
                    "_input_file": input_file_name,

                    "dataset": metadata["dataset"],
                    "llm": metadata["llm"],

                    "source_index": record.get("source_index"),
                    "type": record.get("type"),
                    "question": record.get("question"),
                    "gt": record.get("gt"),
                    "response": record.get("response"),

                    "attempt": record.get("attempt"),
                    "max_tokens_used": record.get("max_tokens_used"),
                    "completion_tokens": record.get("completion_tokens"),
                    "retry_reason": record.get("retry_reason"),

                    **judge_result,
                }

                append_jsonl(out_jsonl, out_obj)

    # Reload after all rounds.
    existing = load_existing_jsonl(out_jsonl)

    ordered = []

    for row_id, record in enumerate(data):
        key = make_result_key(record, row_id)

        if key not in existing:
            raise RuntimeError(f"Missing judgement for key={key} in {input_file_name}")

        ordered.append(existing[key])

    failed = [
        x for x in ordered
        if not is_valid_judgement(x)
    ]

    print("Final failed judgements:", len(failed))

    if failed and RAISE_ON_FAILED_JUDGEMENTS:
        print("First failed judgement:")
        print(json.dumps(failed[0], ensure_ascii=False, indent=2)[:2000])
        raise RuntimeError(
            f"{len(failed)} failed judgements remain in {input_file_name}. "
            f"Re-run this cell or reduce REQUEST_CONCURRENCY."
        )

    if failed:
        print("First failed judgement:")
        print(json.dumps(failed[0], ensure_ascii=False, indent=2)[:2000])
        print("Run the repair cell before computing final accuracy.")

    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(ordered, f, ensure_ascii=False, indent=2)

    print("Saved JSONL:", out_jsonl)
    print("Saved JSON:", out_json)

    return out_json

In [11]:
# cell 11A
# Run and save BM25 judge results for HotpotQA files only.

if "judged_json_paths" not in globals():
    judged_json_paths = []

print("Running HotpotQA BM25 files:")
for input_file_name in HOTPOTQA_INPUT_FILES:
    print("-", input_file_name)

for input_file_name in HOTPOTQA_INPUT_FILES:
    judged_path = judge_file(input_file_name)
    judged_json_paths.append(judged_path)

# Remove duplicates while preserving order.
judged_json_paths = list(dict.fromkeys(judged_json_paths))

print("\nHotpotQA BM25 files judged and saved.")
print("Output dir:", GDRIVE_OUTPUT_DIR)

for path in judged_json_paths:
    print(path)

Running HotpotQA BM25 files:
- hotpotqa_bm25_gemma4_answers.json
- hotpotqa_bm25_gpt_oss_120b_answers.json
- hotpotqa_qwen3.5_bm25_answers.json

File: hotpotqa_bm25_gemma4_answers.json
Dataset: hotpotqa
LLM: gemma4
Round: 1
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Final failed judgements: 0
Saved JSONL: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/hotpotqa_bm25_gemma4_answers__judged.jsonl
Saved JSON: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/hotpotqa_bm25_gemma4_answers__judged.json

File: hotpotqa_bm25_gpt_oss_120b_answers.json
Dataset: hotpotqa
LLM: gpt_oss_120b
Round: 1
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Final failed judgements: 0
Saved JSONL: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/hotpotqa_bm25_gpt_oss_120b_answers__judged.jsonl
Saved JSON: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm

In [12]:
# cell 11B
# Run and save BM25 judge results for 2WikiMultiHopQA files only.

if "judged_json_paths" not in globals():
    judged_json_paths = []

print("Running 2WikiMultiHopQA BM25 files:")
for input_file_name in TWIKIMULTIHOPQA_INPUT_FILES:
    print("-", input_file_name)

for input_file_name in TWIKIMULTIHOPQA_INPUT_FILES:
    judged_path = judge_file(input_file_name)
    judged_json_paths.append(judged_path)

# Remove duplicates while preserving order.
judged_json_paths = list(dict.fromkeys(judged_json_paths))

print("\n2WikiMultiHopQA BM25 files judged and saved.")
print("Output dir:", GDRIVE_OUTPUT_DIR)

for path in judged_json_paths:
    print(path)

Running 2WikiMultiHopQA BM25 files:
- 2wikimultihopqa_bm25_gemma4_answers.json
- 2wikimultihopqa_bm25_gpt_oss_120b_answers.json
- 2wikimultihopqa_qwen3.5_bm25_answers.json

File: 2wikimultihopqa_bm25_gemma4_answers.json
Dataset: 2wikimultihopqa
LLM: gemma4
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging 2wikimultihopqa_bm25_gemma4_answers.json | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


File: 2wikimultihopqa_bm25_gemma4_answers.json
Dataset: 2wikimultihopqa
LLM: gemma4
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Final failed judgements: 0
Saved JSONL: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/2wikimultihopqa_bm25_gemma4_answers__judged.jsonl
Saved JSON: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/2wikimultihopqa_bm25_gemma4_answers__judged.json

File: 2wikimultihopqa_bm25_gpt_oss_120b_answers.json
Dataset: 2wikimultihopqa
LLM: gpt_oss_120b
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging 2wikimultihopqa_bm25_gpt_oss_120b_answers.json | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


File: 2wikimultihopqa_bm25_gpt_oss_120b_answers.json
Dataset: 2wikimultihopqa
LLM: gpt_oss_120b
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Final failed judgements: 0
Saved JSONL: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/2wikimultihopqa_bm25_gpt_oss_120b_answers__judged.jsonl
Saved JSON: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/2wikimultihopqa_bm25_gpt_oss_120b_answers__judged.json

File: 2wikimultihopqa_qwen3.5_bm25_answers.json
Dataset: 2wikimultihopqa
LLM: qwen3.5
Round: 1
Rows: 1000
Valid already judged: 0
Remaining or failed: 1000


Judging 2wikimultihopqa_qwen3.5_bm25_answers.json | round 1:   0%|          | 0/1000 [00:00<?, ?it/s]


File: 2wikimultihopqa_qwen3.5_bm25_answers.json
Dataset: 2wikimultihopqa
LLM: qwen3.5
Round: 2
Rows: 1000
Valid already judged: 1000
Remaining or failed: 0
Final failed judgements: 0
Saved JSONL: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/2wikimultihopqa_qwen3.5_bm25_answers__judged.jsonl
Saved JSON: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/2wikimultihopqa_qwen3.5_bm25_answers__judged.json

2WikiMultiHopQA BM25 files judged and saved.
Output dir: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100
/content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/hotpotqa_bm25_gemma4_answers__judged.json
/content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/hotpotqa_bm25_gpt_oss_120b_answers__judged.json
/content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/hotpotqa

In [13]:
# cell 12
# Load all judged BM25 outputs.
import pandas as pd
import json
from pathlib import Path

# If judged_json_paths is missing because the runtime restarted,
# reload all saved judged JSON files from the output directory.
if "judged_json_paths" not in globals() or len(judged_json_paths) == 0:
    judged_json_paths = sorted(GDRIVE_OUTPUT_DIR.glob("*__judged.json"))

judged_json_paths = [Path(path) for path in judged_json_paths]
judged_json_paths = list(dict.fromkeys(judged_json_paths))

rows = []

for path in judged_json_paths:
    assert path.exists(), f"Judged output not found: {path}"

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    rows.extend(data)

df = pd.DataFrame(rows)

df["judge_score"] = pd.to_numeric(df["judge_score"], errors="coerce")

# Ensure dataset, retrieval, and LLM columns exist even if old judged files are loaded.
metadata_df = df["_input_file"].apply(parse_input_file_name).apply(pd.Series)

for col in ["dataset", "retrieval", "llm"]:
    if col not in df.columns:
        df[col] = metadata_df[col]
    else:
        df[col] = df[col].fillna(metadata_df[col])

print("Total rows:", len(df))
print("Files:", df["_input_file"].nunique())
print("Datasets:", sorted(df["dataset"].dropna().unique()))
print("Retrieval methods:", sorted(df["retrieval"].dropna().unique()))
print("LLMs:", sorted(df["llm"].dropna().unique()))
print("Invalid judgements:", df["judge_score"].isna().sum())

display(df.head())

Total rows: 6000
Files: 6
Datasets: ['2wikimultihopqa', 'hotpotqa']
Retrieval methods: ['bm25']
LLMs: ['gemma4', 'gpt_oss_120b', 'qwen3.5']
Invalid judgements: 0


,_judge_key,_row_id,_input_file,dataset,llm,source_index,type,question,gt,response,attempt,max_tokens_used,completion_tokens,retry_reason,judge_reason,judge_score,judge_raw,judge_error,judge_latency_sec,retrieval
0,0,0,hotpotqa_bm25_gemma4_answers.json,hotpotqa,gemma4,None,comparison,"Which musical fantasy film is older, Bedknobs ...",Bedknobs and Broomsticks,Bedknobs and Broomsticks,None,None,None,None,The candidate response matches the ground trut...,1,"{\n ""reason"": ""The candidate response matches...",None,4.962600,bm25
1,1,1,hotpotqa_bm25_gemma4_answers.json,hotpotqa,gemma4,None,bridge,The Worst Journey in the World is a memoir wri...,Apsley George Benet Cherry-Garrard,Apsley Cherry-Garrard,None,None,None,None,"The candidate correctly identifies the author,...",1,"{\n ""reason"": ""The candidate correctly identi...",None,5.537477,bm25
2,2,2,hotpotqa_bm25_gemma4_answers.json,hotpotqa,gemma4,None,comparison,Which American film director is an advisor for...,John Alan Lasseter,John Lasseter,None,None,None,None,The candidate correctly identifies John Lasset...,1,"{\n ""reason"": ""The candidate correctly identi...",None,5.410217,bm25
3,3,3,hotpotqa_bm25_gemma4_answers.json,hotpotqa,gemma4,None,comparison,Were the documentaries One Big Hapa Family an...,no,Information not available,None,None,None,None,The candidate claims information is unavailabl...,0,"{\n ""reason"": ""The candidate claims informati...",None,6.511005,bm25
4,4,4,hotpotqa_bm25_gemma4_answers.json,hotpotqa,gemma4,None,bridge,Who is the writer of this song that was inspir...,Phil Spector,Phil Spector,None,None,None,None,The candidate response matches the ground trut...,1,"{\n ""reason"": ""The candidate response matches...",None,6.671822,bm25


In [14]:
#cell 13
# Compute accuracy overall and by type.
valid_df = df[df["judge_score"].isin([0, 1])].copy()

if len(valid_df) != len(df):
    raise RuntimeError("Some judgements are invalid. Fix them before computing accuracy.")

def accuracy_table(dataframe, group_cols):
    # Build accuracy table.
    result = (
        dataframe
        .groupby(group_cols, dropna=False)
        .agg(
            total=("judge_score", "size"),
            correct=("judge_score", "sum"),
            accuracy=("judge_score", "mean"),
        )
        .reset_index()
    )

    result["correct"] = result["correct"].astype(int)
    result["accuracy_percent"] = result["accuracy"] * 100.0

    return result.sort_values(group_cols).reset_index(drop=True)

# Accuracy for each of the six files.
per_file_overall = accuracy_table(valid_df, ["dataset", "llm", "_input_file"])

# Accuracy for each file split by question type.
per_file_type = accuracy_table(valid_df, ["dataset", "llm", "_input_file", "type"])

# Accuracy for each dataset across all LLMs.
per_dataset_overall = accuracy_table(valid_df, ["dataset"])

# Accuracy for each dataset split by question type.
per_dataset_type = accuracy_table(valid_df, ["dataset", "type"])

# Accuracy for each LLM across both datasets.
per_llm_overall = accuracy_table(valid_df, ["llm"])

# Accuracy for each LLM split by question type across both datasets.
per_llm_type = accuracy_table(valid_df, ["llm", "type"])

# Combined type accuracy across all six files.
combined_type = accuracy_table(valid_df, ["type"])

# Combined overall accuracy across all six files.
combined_overall = pd.DataFrame([
    {
        "scope": "all_files_combined",
        "total": int(valid_df["judge_score"].size),
        "correct": int(valid_df["judge_score"].sum()),
        "accuracy": float(valid_df["judge_score"].mean()),
        "accuracy_percent": float(valid_df["judge_score"].mean() * 100.0),
    }
])

print("Per-file overall accuracy:")
display(per_file_overall)

print("Per-file per-type accuracy:")
display(per_file_type)

print("Per-dataset overall accuracy:")
display(per_dataset_overall)

print("Per-dataset per-type accuracy:")
display(per_dataset_type)

print("Per-LLM overall accuracy:")
display(per_llm_overall)

print("Per-LLM per-type accuracy:")
display(per_llm_type)

print("Combined per-type accuracy:")
display(combined_type)

print("Combined overall accuracy:")
display(combined_overall)

Per-file overall accuracy:


,dataset,llm,_input_file,total,correct,accuracy,accuracy_percent
0,2wikimultihopqa,gemma4,2wikimultihopqa_bm25_gemma4_answers.json,1000,365,0.365,36.5
1,2wikimultihopqa,gpt_oss_120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,1000,372,0.372,37.2
2,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,1000,447,0.447,44.7
3,hotpotqa,gemma4,hotpotqa_bm25_gemma4_answers.json,1000,508,0.508,50.8
4,hotpotqa,gpt_oss_120b,hotpotqa_bm25_gpt_oss_120b_answers.json,1000,586,0.586,58.6
5,hotpotqa,qwen3.5,hotpotqa_qwen3.5_bm25_answers.json,1000,637,0.637,63.7


Per-file per-type accuracy:


,dataset,llm,_input_file,type,total,correct,accuracy,accuracy_percent
0,2wikimultihopqa,gemma4,2wikimultihopqa_bm25_gemma4_answers.json,bridge_comparison,250,54,0.216000,21.600000
1,2wikimultihopqa,gemma4,2wikimultihopqa_bm25_gemma4_answers.json,comparison,250,195,0.780000,78.000000
2,2wikimultihopqa,gemma4,2wikimultihopqa_bm25_gemma4_answers.json,compositional,250,34,0.136000,13.600000
3,2wikimultihopqa,gemma4,2wikimultihopqa_bm25_gemma4_answers.json,inference,250,82,0.328000,32.800000
4,2wikimultihopqa,gpt_oss_120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,bridge_comparison,250,25,0.100000,10.000000
5,2wikimultihopqa,gpt_oss_120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,comparison,250,192,0.768000,76.800000
6,2wikimultihopqa,gpt_oss_120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,compositional,250,49,0.196000,19.600000
7,2wikimultihopqa,gpt_oss_120b,2wikimultihopqa_bm25_gpt_oss_120b_answers.json,inference,250,106,0.424000,42.400000
8,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,bridge_comparison,250,88,0.352000,35.200000
9,2wikimultihopqa,qwen3.5,2wikimultihopqa_qwen3.5_bm25_answers.json,comparison,250,208,0.832000,83.200000


Per-dataset overall accuracy:


...

,llm,type,total,correct,accuracy,accuracy_percent
0,gemma4,bridge,700,344,0.491429,49.142857
1,gemma4,bridge_comparison,250,54,0.216000,21.600000
2,gemma4,comparison,550,359,0.652727,65.272727
3,gemma4,compositional,250,34,0.136000,13.600000
4,gemma4,inference,250,82,0.328000,32.800000
5,gpt_oss_120b,bridge,700,417,0.595714,59.571429
6,gpt_oss_120b,bridge_comparison,250,25,0.100000,10.000000
7,gpt_oss_120b,comparison,550,361,0.656364,65.636364
8,gpt_oss_120b,compositional,250,49,0.196000,19.600000
9,gpt_oss_120b,inference,250,106,0.424000,42.400000


Combined per-type accuracy:


,type,total,correct,accuracy,accuracy_percent
0,bridge,2100,1174,0.559048,55.904762
1,bridge_comparison,750,167,0.222667,22.266667
2,comparison,1650,1152,0.698182,69.818182
3,compositional,750,141,0.188000,18.800000
4,inference,750,281,0.374667,37.466667


Combined overall accuracy:


,scope,total,correct,accuracy,accuracy_percent
0,all_files_combined,6000,2915,0.485833,48.583333


In [15]:
#cell 14
# Save accuracy summaries.
SUMMARY_OUTPUT_DIR = GDRIVE_OUTPUT_DIR / "accuracy_summaries"
SUMMARY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_tables = {
    "per_file_overall": per_file_overall,
    "per_file_type": per_file_type,
    "per_dataset_overall": per_dataset_overall,
    "per_dataset_type": per_dataset_type,
    "per_llm_overall": per_llm_overall,
    "per_llm_type": per_llm_type,
    "combined_type": combined_type,
    "combined_overall": combined_overall,
}

for name, table in summary_tables.items():
    csv_path = SUMMARY_OUTPUT_DIR / f"{name}.csv"
    json_path = SUMMARY_OUTPUT_DIR / f"{name}.json"

    table.to_csv(csv_path, index=False)
    table.to_json(json_path, orient="records", force_ascii=False, indent=2)

    print("Saved:", csv_path)
    print("Saved:", json_path)

print("Summary output dir:", SUMMARY_OUTPUT_DIR)

Saved: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/accuracy_summaries/per_file_overall.csv
Saved: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/accuracy_summaries/per_file_overall.json
Saved: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/accuracy_summaries/per_file_type.csv
Saved: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/accuracy_summaries/per_file_type.json
Saved: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/accuracy_summaries/per_dataset_overall.csv
Saved: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/accuracy_summaries/per_dataset_overall.json
Saved: /content/drive/MyDrive/final_project/BM25/answer/bm25/llm_judge_qwen35_27b_accuracy_a100/accuracy_summaries/per_dataset_type.csv
Saved: /content/drive/MyDrive/final_project/B